In [1]:
!pip install -q langchain-community pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import re
import pandas as pd
import os
import io

from google.colab import auth
from googleapiclient.discovery import build
from google.auth import default
from googleapiclient.http import MediaIoBaseDownload

FILE_ID = "1ZfMR_y92P6stVVupl1QGykcknl6EJU8X"
pdf_path = "/content/npr-7150-2d.pdf"

if not os.path.exists(pdf_path):
    auth.authenticate_user()

    creds, _ = default()
    drive = build("drive", "v3", credentials=creds)

    request = drive.files().get_media(
        fileId=FILE_ID,
        supportsAllDrives=True
    )

    with io.FileIO(pdf_path, "wb") as f:
        downloader = MediaIoBaseDownload(f, request)

        done = False
        while not done:
            status, done = downloader.next_chunk()
            print(f"Downloaded {int(status.progress() * 100)}%")

    print(f"PDF downloaded to: {pdf_path}")

else:
    print(f"PDF already exists at: {pdf_path}")

Downloaded 100%
PDF downloaded to: /content/npr-7150-2d.pdf


In [3]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(pdf_path)
pages = loader.load()

print(f"Loaded {len(pages)} pages")

/tmp/ipykernel_2907/3021515166.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 89 pages


## Exploration of NPR 7150.2D Structure

Before transforming the standard into individual SWE requirements,
we examine the original document structure to determine which
information should be retained for downstream retrieval and auditing.

### 1. Inspect Raw PDF Extraction

We first inspect several pages from the original PDF extraction to understand
the document structure and identify repeated metadata, headings, notes,
requirements, and other patterns that may affect parsing.

In [4]:
for page_num in [0, 10, 19, 34, 45]:
    print("=" * 80)
    print(f"PAGE {page_num + 1}")
    print("=" * 80)
    print(pages[page_num].page_content[:2500])
    print()

PAGE 1
| NODIS Library | Program Formulation(7000s) | Search | 
 NASA
Procedural
Requirements 
NPR 7150.2D 
Effective Date: March 08, 2022
Expiration Date: March 08, 2027
COMPLIANCE IS MANDATORY FOR NASA EMPLOYEES 
NASA Software Engineering Requirements
Responsible Office: Office of the Chief Engineer
Table of Contents 
Preface
P.1 Purpose
P.2 Applicability
P.3 Authority
P.4 Applicable Documents and Forms
P.5 Measurement/Verification
P.6 Cancellation 
Chapter 1. Introduction
1.1 Overview
1.2 Hierarchy of NASA Software-Related Engineering and Program/Project Documents
1.3 Document Structure 
Chapter 2. Roles, Responsibilities, and Principles Related to
Tailoring of the Requirements 
2.1 Roles and Responsibilities 
2.2 Principles Related to Tailoring of the Requirements 
Chapter 3. Software Management Requirements
3.1 Software Life Cycle Planning
3.2 Software Cost Estimation
3.3 Software Schedules
3.4 Software Training
NPR 7150.2D -- TOC
This document does not bind the public, except as 

### 2. Identify Common Document Structures

The standard contains multiple forms of structured information. We inspect
their frequency before deciding what should be preserved in the final dataset.

In [5]:
raw_text = "\n".join(page.page_content for page in pages)

patterns = {
    "SWE requirement IDs": r"\[SWE-\d+\]",
    "Notes": r"\bNote:",
    "Chapter headings": r"(?m)^Chapter\s+\d+",
    "Numbered sections": r"(?m)^\d+\.\d+(?:\.\d+)*\s+",
    "Lettered subclauses": r"(?m)^[a-z]\.\s+",
    "Numbered subclauses": r"(?m)^\(\d+\)\s+"
}

for name, pattern in patterns.items():
    matches = re.findall(pattern, raw_text)
    print(f"{name}: {len(matches)}")

SWE requirement IDs: 130
Notes: 57
Chapter headings: 12
Numbered sections: 380
Lettered subclauses: 307
Numbered subclauses: 22


### 3. Inspect Chapter Organization

NPR 7150.2D is organized by chapter and section. Since later
retrieval may originate from chapter-level context, we inspect and preserve
this hierarchy.

In [6]:
chapter_pattern = re.compile(
    r"(?m)^Chapter\s+(\d+)[:.]\s*(.*)"
)

chapter_matches = chapter_pattern.findall(raw_text)

for chapter_num, title in chapter_matches:
    print(f"Chapter {chapter_num}: {title}")

Chapter 1: Introduction
Chapter 2: Roles, Responsibilities, and Principles Related to
Chapter 3: Software Management Requirements
Chapter 4: Software Engineering (Life Cycle) Requirements
Chapter 5: Supporting Software Life Cycle Requirements
Chapter 6: Recommended Software Documentation Contents
Chapter 1: Introduction
Chapter 2: Roles, Responsibilities, and
Chapter 3: Software Management
Chapter 4: Software Engineering Life Cycle
Chapter 5: Supporting Software Life Cycle
Chapter 6: Recommended Software Records


The raw extraction contains duplicate chapter headings because the Table of
Contents and chapter body headings are both present. Some titles also span
multiple PDF lines. These will be normalized during transformation.

### 4. Text Cleaning and Page Preservation

The PDF text contains repeated headers, footers, page numbers, and document metadata.
These are removed before exploration so they do not interfere with section detection
or requirement extraction. Page boundaries are preserved so extracted clauses can
still be traced back to their source page

In [7]:
full_text = ""
page_ranges = []

for page_num, page in enumerate(pages, start=1):
    text = page.page_content

    text = re.sub(
        r"NPR 7150\.2D -- [^\n]+\n"
        r"This document does not bind the public.*?"
        r"Page\s+\d+\s+of\s+\d+",
        "",
        text,
        flags=re.DOTALL
    )

    text = re.sub(
        r"NPR 7150\.2D -- [^\n]+ Page\s+\d+\s+of\s+\d+",
        "",
        text
    )

    start = len(full_text)

    full_text += text.strip() + "\n\n"

    end = len(full_text)

    page_ranges.append({
        "page": page_num,
        "start": start,
        "end": end
    })

print(full_text[:2000])

| NODIS Library | Program Formulation(7000s) | Search | 
 NASA
Procedural
Requirements 
NPR 7150.2D 
Effective Date: March 08, 2022
Expiration Date: March 08, 2027
COMPLIANCE IS MANDATORY FOR NASA EMPLOYEES 
NASA Software Engineering Requirements
Responsible Office: Office of the Chief Engineer
Table of Contents 
Preface
P.1 Purpose
P.2 Applicability
P.3 Authority
P.4 Applicable Documents and Forms
P.5 Measurement/Verification
P.6 Cancellation 
Chapter 1. Introduction
1.1 Overview
1.2 Hierarchy of NASA Software-Related Engineering and Program/Project Documents
1.3 Document Structure 
Chapter 2. Roles, Responsibilities, and Principles Related to
Tailoring of the Requirements 
2.1 Roles and Responsibilities 
2.2 Principles Related to Tailoring of the Requirements 
Chapter 3. Software Management Requirements
3.1 Software Life Cycle Planning
3.2 Software Cost Estimation
3.3 Software Schedules
3.4 Software Training

3.4 Software Training
3.5 Software Classification Assessments
3.6 Software 

### 5. Validate Cleaned Text

After removing repeated headers and footers, we inspect the cleaned text to
confirm that meaningful chapter, section, requirement, note, and subclause
content has been preserved.

In [8]:
print(full_text[full_text.find("Chapter 3:"):full_text.find("Chapter 3:") + 3000])

Chapter 3: Software Management
Requirements
3.1 Software Life Cycle Planning
3.1.1 Software life cycle planning covers the software aspects of a project from inception through
retirement. The software life cycle planning is an organizing process that considers the software as a
whole and provides the planning activities required to ensure a coordinated, well-engineered process
for defining and implementing project activities. These processes, plans, and activities are
coordinated within the project. At project conception, software needs for the project are analyzed,
including acquisition, supply, development, operation, maintenance, retirement, decommissioning,
and supporting activities and processes. The software effort is scoped, the development processes
defined, measurements defined, and activities are documented in software planning documents. 
3.1.2 The project manager shall assess options for software acquisition versus development.
[SWE-033] 
Note: The assessment can include ri

In [9]:
print("Raw characters:", len(raw_text))
print("Cleaned characters:", len(full_text))
print("Characters removed:", len(raw_text) - len(full_text))

Raw characters: 212308
Cleaned characters: 178054
Characters removed: 34254


In [10]:
section_pattern = re.compile(
    r"(?m)^([2-5]\.\d+(?:\.\d+)*)\s+"
)

section_matches = list(section_pattern.finditer(full_text))

print(f"Found {len(section_matches)} numbered sections")

Found 351 numbered sections


In [11]:
for match in section_matches[:20]:
    print(match.group(1), "->", full_text[match.start():match.start() + 150].replace("\n", " "))

2.1 -> 2.1 Roles and Responsibilities  2.2 Principles Related to Tailoring of the Requirements  Chapter 3. Software Management Requirements 3.1 Software Life
2.2 -> 2.2 Principles Related to Tailoring of the Requirements  Chapter 3. Software Management Requirements 3.1 Software Life Cycle Planning 3.2 Software Cos
3.1 -> 3.1 Software Life Cycle Planning 3.2 Software Cost Estimation 3.3 Software Schedules 3.4 Software Training  3.4 Software Training 3.5 Software Classif
3.2 -> 3.2 Software Cost Estimation 3.3 Software Schedules 3.4 Software Training  3.4 Software Training 3.5 Software Classification Assessments 3.6 Software 
3.3 -> 3.3 Software Schedules 3.4 Software Training  3.4 Software Training 3.5 Software Classification Assessments 3.6 Software Assurance and Software Indepe
3.4 -> 3.4 Software Training  3.4 Software Training 3.5 Software Classification Assessments 3.6 Software Assurance and Software Independent Verification & Va
3.4 -> 3.4 Software Training 3.5 Software Classifica

In [12]:
def get_page_number(offset):
    for page_info in page_ranges:
        if page_info["start"] <= offset < page_info["end"]:
            return page_info["page"]

    return None

In [13]:
pd.set_option("display.max_colwidth", 200)

### 6. Inspect Representative Sections

Not every numbered section has the same structure. Some contain a single SWE
requirement, while others include notes, lists, or nested conditions. We inspect
examples before reducing the document into requirement-level records.

In [14]:
examples = ["2.1.5.10", "3.1.14", "4.5.3", "4.5.9"]

for section_id in examples:
    match = next(
        m for m in section_matches
        if m.group(1) == section_id
        and get_page_number(m.start()) > 10
    )

    start = match.start()

    later_matches = [
        m for m in section_matches
        if m.start() > start
    ]

    end = later_matches[0].start() if later_matches else len(full_text)

    print("=" * 80)
    print("SECTION:", section_id)
    print("=" * 80)
    print(full_text[start:end][:1500])

SECTION: 2.1.5.10
2.1.5.10 For Class A, B, and C software projects, each Center Director, or designee, shall establish
and maintain software cost repository(ies) that contains at least the following measures: [SWE-142]

a. Planned and actual effort and cost. 
b. Planned and actual schedule dates for major milestones. 
c. Both planned and actual values for key cost parameters that typically include software size,
requirements count, defects counts for maintenance or sustaining engineering projects, and cost
model inputs. 
d. Project descriptors or metadata that typically includes software class, software domain/type, and
requirements volatility. 

SECTION: 3.1.14
3.1.14 The project manager shall satisfy the following conditions when a COTS, GOTS, MOTS,
OSS, or reused software component is acquired or used: [SWE-027] 
a. The requirements to be met by the software component are identified. 
b. The software component includes documentation to fulfill its intended purpose (e.g., usage
instr

Inspection of representative sections revealed occasional duplicated text at
PDF page boundaries. For example, part of the explanatory text associated with
SWE-189 appears twice after extraction. This indicates that additional
page-boundary cleanup may be necessary before the parsed text is used for
retrieval.

## Structured SWE Requirement Dataset

Based on the document exploration above, individual SWE requirements are used
as the primary records for the initial dataset. Chapter, section, and page
metadata are retained for traceability and later retrieval.

In [15]:
rows = []

for i, match in enumerate(section_matches):
    section = match.group(1)

    start = match.end()

    if i + 1 < len(section_matches):
        end = section_matches[i + 1].start()
    else:
        end = len(full_text)

    block_text = full_text[start:end].strip()

    swe_match = re.search(r"\[SWE-(\d+)\]", block_text)

    if not swe_match:
        continue

    swe_id = f"SWE-{swe_match.group(1)}"

    requirement_text = re.sub(
        r"\[SWE-\d+\]",
        "",
        block_text
    ).strip()

    requirement_text = re.split(
        r"\nChapter\s+\d+[:.]",
        requirement_text
    )[0].strip()

    page = get_page_number(match.start())

    rows.append({
        "swe_id": swe_id,
        "section": section,
        "requirement_text": requirement_text,
        "page": page
    })

requirements_df = pd.DataFrame(rows)

print(f"Found {len(requirements_df)} SWE requirements")
print("Total rows:", len(requirements_df))
print("Unique SWE IDs:", requirements_df["swe_id"].nunique())
print("Unique sections:", requirements_df["section"].nunique())

Found 130 SWE requirements
Total rows: 130
Unique SWE IDs: 130
Unique sections: 130


In [16]:
requirements_df.isna().sum()

,0
swe_id,0
section,0
requirement_text,0
page,0


In [17]:
requirements_df[
    requirements_df["swe_id"].duplicated(keep=False)
]

,swe_id,section,requirement_text,page


In [18]:
requirements_df

,swe_id,section,requirement_text,page
0,SWE-002,2.1.1.1,The NASA OCE shall lead and maintain a NASA Software Engineering Initiative to advance\nsoftware engineering practices.,11
1,SWE-004,2.1.1.2,The NASA OCE shall periodically benchmark each Center’s software engineering capability\nagainst requirements in this directive. \nNote: Capability Maturity Model® Integration (CMMI®) for Develop...,11
2,SWE-152,2.1.1.3,The NASA OCE shall periodically review the project requirements mapping matrices.,11
3,SWE-129,2.1.1.4,The NASA OCE shall authorize appraisals against selected requirements in this NPR to\ncheck compliance.,11
4,SWE-100,2.1.1.5,The NASA OCE and Center training organizations shall provide training to advance\nsoftware engineering practices.,11
...,...,...,...,...
125,SWE-200,5.4.6,"The project manager shall collect, track, and report software requirements volatility metrics.",42
126,SWE-201,5.5.1,The project manager shall track and maintain software non-conformances (including defects in\ntools and appropriate ground software).,42
127,SWE-202,5.5.2,"The project manager shall define and implement clear software severity levels for all software\nnon-conformances (including tools, COTS, GOTS, MOTS, OSS, reused software components, and\napplicabl...",42
128,SWE-203,5.5.3,"The project manager shall implement mandatory assessments of reported non-conformances\nfor all COTS, GOTS, MOTS, OSS, and/or reused software components. \n\nNote: This includes operating systems,...",42


In [19]:
requirements_df["chapter"] = (
    requirements_df["section"]
    .str.split(".")
    .str[0]
)

requirements_df["chapter"].value_counts().sort_index()

,count
chapter,
2,30
3,45
4,34
5,21


In [20]:
requirements_df["section_group"] = (
    requirements_df["section"]
    .str.extract(r"^(\d+\.\d+)")[0]
)

requirements_df.groupby(
    ["chapter", "section_group"]
).size()

chapter  section_group
2        2.1              28
         2.2               2
3        3.1              13
         3.10              2
         3.11              7
         3.12              1
         3.2               3
         3.3               3
         3.4               1
         3.5               2
         3.6               5
         3.7               5
         3.8               2
         3.9               1
4        4.1               6
         4.2               2
         4.3               1
         4.4               7
         4.5              13
         4.6               5
5        5.1               8
         5.2               1
         5.3               3
         5.4               5
         5.5               4
dtype: int64

In [22]:
requirements_df["word_count"] = (
    requirements_df["requirement_text"]
    .str.split()
    .str.len()
)

requirements_df["word_count"].describe()

,word_count
count,130.000000
mean,63.015385
std,69.136085
min,10.000000
25%,20.250000
50%,38.000000
75%,85.000000
max,412.000000


In [23]:
requirements_df.nlargest(
    10,
    "word_count"
)[
    ["swe_id", "section", "page", "word_count", "requirement_text"]
]

,swe_id,section,page,word_count,requirement_text
25,SWE-214,2.1.5.16,14,412,"The Center Director or designee (e.g., the Civil Servant Technical POC for the software\nproduct) shall perform the following actions for each type of internal NASA software transfer or\nreuse: \..."
27,SWE-126,2.1.8.2,16,374,The technical and institutional authorities for requirements in this directive shall: \na. Assess projects’ requirements mapping matrices and tailoring from requirements in this directive\nby:\n\...
64,SWE-032,3.9.2,28,305,"The project manager shall acquire, develop, and maintain software from an organization with a\nnon-expired CMMI®-DEV rating as measured by a CMMI® Institute Certified Lead Appraiser as\nfollows: ..."
66,SWE-148,3.10.2,29,297,The project manager shall evaluate software for potential reuse by other projects across\nNASA and contribute reuse candidates to the appropriate NASA internal sharing and reuse software\nsystem. ...
42,SWE-027,3.1.14,22,236,"The project manager shall satisfy the following conditions when a COTS, GOTS, MOTS,\nOSS, or reused software component is acquired or used: \na. The requirements to be met by the software compone..."
59,SWE-134,3.7.3,26,216,"If a project has safety-critical software or mission-critical software, the project manager shall\nimplement the following items in the software: \na. The software is initialized, at first start ..."
98,SWE-189,4.5.9,36,201,"The project manager shall ensure that the code coverage measurements for the software are\nselected, implemented, tracked, recorded, and reported. \nNote: This requirement can be met by running u..."
50,SWE-020,3.5.1,24,177,"The project manager shall classify each system and subsystem containing software in\naccordance with the highest applicable software classification definitions for Classes A, B, C, D, E,\nand F so..."
41,SWE-125,3.1.13,22,171,"Each project manager with software components shall maintain a requirements mapping\nmatrix or multiple requirements mapping matrices against requirements in this NPR, including those\ndelegated t..."
16,SWE-006,2.1.5.6,13,144,"Center Director, or designee, shall maintain a reliable list of their Center’s programs and\nprojects containing Class A, B, C, and D software. The list should include: \na. Project/program name ..."


Exploration of NPR 7150.2D shows that the document is hierarchical rather
than a flat list of requirements. In addition to 130 SWE requirement IDs,
the standard contains chapter and section headings, explanatory notes,
lettered and numbered subclauses, and other supporting context.

Based on these findings, we retain chapter, section, page, and parent-section
metadata alongside individual SWE requirements. Larger section-level text
blocks are also preserved for downstream retrieval because some compliance
decisions may require context beyond an isolated SWE statement.

Manual inspection also identified occasional PDF extraction artifacts at page
boundaries, which will require additional cleanup or validation.